# 00 · Quickstart: pipelines, configs, tracking
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/00_quickstart.ipynb)

Run a pipeline, change a config, see it in MLflow. Every lesson works this way.

In [ ]:
# Setup: works in Colab, the dev container and a local clone.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


## 1. Run a pipeline
The GLM and the homogeneous model are closed-form: this takes about a minute on the full 678k-policy dataset.

In [ ]:
run("frequency", models=["homogeneous", "glm"])
leaderboard("frequency", ["homogeneous", "glm"])

## 2. Reconcile to the published benchmark
Each model has a reconciliation table: **PASS** means this run reproduces the published result within tolerance.

In [ ]:
load("glm.freq_reconciliation")

## 3. Change a config, re-run
Parameters live in `conf/base/parameters_*.yml`. Override any key at run time, e.g. a different GLM formula:

In [ ]:
formula = "ClaimNb ~ DrivAgeGLM + BonusMalusGLM + VehBrand + VehGas + DensityGLM + AreaGLM"
run("frequency", models=["glm"], params={} if FAST else {"frequency.models.glm.formula": formula})
leaderboard("frequency", ["glm"])

## 4. Scale experiments
`frequency.data.scale` subsamples the data: check how a model's out-of-sample deviance and stability change with data volume.

In [ ]:
rows = []
for s in ([0.02] if FAST else [0.05, 0.25, 1.0]):
    run("frequency", models=["glm"], params={"frequency.data.scale": s})
    rows.append({"scale": s, **load("glm.freq_metrics")})
import pandas as pd; pd.DataFrame(rows)[["scale", "deviance_learn", "deviance_test", "gini_test"]]

## 5. Tracking
Every run above is in MLflow (params, metrics, figures, model pickle, benchmark table). Query it from Python below, or open the UI with `mlflow_ui()` in the last cell (works in Colab, the container and locally).

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.search_runs(experiment_names=["deep-actuarial-science"]).filter(regex="start_time|metrics.glm.deviance").head()

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()